# High-Liquidity Symbols By 10-Day Average Volume

This notebook loads daily bars from MongoDB, computes each symbol's trailing 10-trading-day average volume, keeps the latest available result per symbol, filters symbols with average volume greater than 1,000,000, and saves the result to `high_liquid.csv`.

Assumption: "symbols with 10 day average volume > 1 million" means the latest available trailing 10-day average volume per symbol, not every historical row above the threshold.

## Setup

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

PROJECT_ROOT

In [ ]:
import pandas as pd

from loaders.data_loader import list_symbols, load_market_bars

## Parameters

In [ ]:
WINDOW = 10
MIN_AVG_VOLUME = 1_000_000
OUTPUT_PATH = PROJECT_ROOT / "scripts" / "high_liquid.csv"

## Load Market Bars

The loader uses read-only MongoDB `distinct` and `find` operations.

In [ ]:
def load_all_bars() -> pd.DataFrame:
    symbols = list_symbols()
    assert symbols, "No symbols found in MongoDB collection."

    bars = load_market_bars(symbols=symbols)
    assert not bars.empty, "No market bars returned from MongoDB."
    return bars


bars = load_all_bars()

{
    "symbol_count": bars["symbol"].nunique(),
    "row_count": len(bars),
    "columns": bars.columns.tolist(),
}

## Prepare Volume Data

Sorting by `symbol, time` before rolling prevents cross-symbol leakage and makes the trailing window deterministic.

In [ ]:
def prepare_volume_data(bars: pd.DataFrame) -> pd.DataFrame:
    required_columns = {"symbol", "time", "volume"}
    missing_columns = required_columns.difference(bars.columns)
    assert not missing_columns, f"Missing required columns: {sorted(missing_columns)}"

    volume_data = bars.loc[:, ["symbol", "time", "volume"]].copy()
    volume_data["time"] = pd.to_datetime(volume_data["time"])
    volume_data["volume"] = pd.to_numeric(volume_data["volume"], errors="coerce")
    volume_data = volume_data.dropna(subset=["symbol", "time", "volume"])
    return volume_data.sort_values(["symbol", "time"]).reset_index(drop=True)


volume_data = prepare_volume_data(bars)
volume_data.head()

## Compute 10-Day Average Volume

In [ ]:
def add_average_volume(volume_data: pd.DataFrame, window: int) -> pd.DataFrame:
    result = volume_data.copy()
    result[f"avg_volume_{window}d"] = (
        result.groupby("symbol", group_keys=False)["volume"]
        .rolling(window=window, min_periods=window)
        .mean()
        .reset_index(level=0, drop=True)
    )
    return result


volume_with_average = add_average_volume(volume_data, WINDOW)
volume_with_average.tail()

## Filter High-Liquidity Symbols

In [ ]:
def filter_high_liquidity(
    volume_with_average: pd.DataFrame,
    min_avg_volume: float,
    window: int,
) -> pd.DataFrame:
    avg_column = f"avg_volume_{window}d"
    latest_volume = (
        volume_with_average.dropna(subset=[avg_column])
        .groupby("symbol", group_keys=False)
        .tail(1)
        .sort_values(avg_column, ascending=False)
        .reset_index(drop=True)
    )

    return latest_volume.loc[
        latest_volume[avg_column] > min_avg_volume,
        ["symbol", "time", "volume", avg_column],
    ].reset_index(drop=True)


high_liquid = filter_high_liquidity(volume_with_average, MIN_AVG_VOLUME, WINDOW)
high_liquid

## Save Result

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
high_liquid.to_csv(OUTPUT_PATH, index=False)

{
    "high_liquid_count": len(high_liquid),
    "latest_input_date": high_liquid["time"].max().date().isoformat() if not high_liquid.empty else None,
    "output_path": str(OUTPUT_PATH),
}